# 📊 Model Evaluation — Qwen2.5-7B Itemset Extractor

Evaluates `OliverSlivka/qwen2.5-7b-itemset-extractor` against Apriori ground truth.

**Eval datasets:** 30 dedicated datasets from `OliverSlivka/itemset-eval-v2` — NEVER used in training.

**Workflow:**
1. **Cell 1** — Install deps (run once, then restart kernel)
2. **Cell 2** — Config: model path, number of eval datasets
3. **Cell 3** — Environment check (suppress TF/JAX)
4. **Cell 4** — Load model (Unsloth/transformers)
5. **Cell 5** — Load eval datasets from HuggingFace (with pre-computed ground truth)
6. **Cell 6** — Define helper functions (Apriori, inference, metrics)
7. **Cell 7** — Run evaluation loop
8. **Cell 8** — Display results + summary + pass/fail verdict
9. **Cell 9** — (Optional) Inspect raw model output for any dataset
10. **Cell 10** — Save results to files

**Metrics:** Precision · Recall · F1 · Exact Match · JSON Parse Rate · Hallucination · Inference Time

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
# Same idempotent install as training notebook.
# ⚠️  Run ONCE, then Restart Kernel, then run from Cell 2.

import os, glob, shutil, subprocess

USER_SP = os.path.expanduser("~/.local/lib/python3.12/site-packages")

# Only install if unsloth is missing
unsloth_dir = os.path.join(USER_SP, "unsloth")
if not os.path.isdir(unsloth_dir):
    print("📦 First run — installing ML packages...")
    subprocess.check_call(
        "pip install unsloth trl datasets transformers accelerate "
        "bitsandbytes huggingface_hub peft safetensors sentencepiece protobuf -q".split()
    )
    print("📦 Install complete.")
else:
    print("✅ Packages already installed — skipping pip install")

# Remove ONLY core torch + nvidia (keep torchvision/torchaudio)
removed = []
for pattern in ["torch", "torch-*"]:
    for p in glob.glob(os.path.join(USER_SP, pattern)):
        basename = os.path.basename(p)
        if basename.startswith("torchvision") or basename.startswith("torchaudio"):
            continue
        shutil.rmtree(p, ignore_errors=True)
        removed.append(basename)

for p in glob.glob(os.path.join(USER_SP, "nvidia*")):
    shutil.rmtree(p, ignore_errors=True)
    removed.append(os.path.basename(p))

if removed:
    print(f"🗑️  Cleaned: {removed}")
else:
    print("✅ No user-level torch/nvidia to clean")

print("\n" + "=" * 60)
print("⚠️  RESTART THE KERNEL → then run Cell 2 onwards")
print("=" * 60)

In [ ]:
# ── Cell 2: CONFIG — edit these values ────────────────────────────────────────

# Model to evaluate (HF repo or local path)
MODEL_PATH = "OliverSlivka/qwen2.5-7b-itemset-extractor"

# ── Eval dataset (dedicated, separate from training) ─────────────────────────
# These 30 datasets are NEVER used in training → fair evaluation
# Hosted at: https://huggingface.co/datasets/OliverSlivka/itemset-eval-v2
HF_EVAL_DATASET = "OliverSlivka/itemset-eval-v2"

# Number of datasets to evaluate (max 30)
N_EVAL     = 20           # increase up to 30 for more reliable results

# Apriori parameters (must match what the model was trained on)
MIN_SUPPORT = 3
MAX_SIZE    = 3

# Generation config
MAX_NEW_TOKENS = 2048
TEMPERATURE    = 0.1

# Output directory for saved results
OUTPUT_DIR = "eval_results"

# Random seed for reproducibility
SEED = 42

print("Config loaded ✅")
print(f"  Model:          {MODEL_PATH}")
print(f"  Eval dataset:   {HF_EVAL_DATASET}")
print(f"  N eval:         {N_EVAL}")
print(f"  Min support:    {MIN_SUPPORT}")

In [ ]:
# ── Cell 3: Environment check + suppress TF/JAX (prevents Keras 3 crash) ─────
import os
os.environ["USE_TF"]  = "0"
os.environ["USE_JAX"] = "0"

import torch
import json, re, time, random, itertools
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.version.cuda}")
print(f"GPU avail: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Cell 4: Load model ────────────────────────────────────────────────────────
# Tries Unsloth first (faster on H200), falls back to plain transformers.
# Loads the MERGED 4-bit model directly — NOT an adapter.

try:
    from unsloth import FastLanguageModel

    print(f"📥 Loading via Unsloth: {MODEL_PATH}")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_PATH,
        max_seq_length=4096,
        load_in_4bit=True,
        dtype=None,
    )
    FastLanguageModel.for_inference(model)
    print("✅ Loaded (Unsloth, 4-bit)")

except (ImportError, Exception) as e:
    print(f"Unsloth unavailable ({e}) — using transformers...")
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

    if torch.cuda.is_available():
        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_quant_type="nf4",
        )
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_PATH, quantization_config=bnb,
            device_map="auto", trust_remote_code=True,
        )
        print(f"✅ Loaded (transformers, 4-bit on GPU)")
    else:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_PATH, torch_dtype=torch.float32,
            device_map="cpu", trust_remote_code=True,
        )
        print("⚠️  Loaded on CPU (will be slow)")

    model.eval()

print(f"\nGPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB" if torch.cuda.is_available() else "")

In [ ]:
# ── Cell 5: Load eval datasets from HuggingFace ──────────────────────────────
# Dedicated eval dataset — 30 datasets, NEVER used in training, with ground truth

random.seed(SEED)

from datasets import load_dataset

def parse_csv_text_to_transactions(csv_text: str) -> list:
    """Convert 'Row N: item1, item2' format back into transaction lists."""
    transactions = []
    for line in csv_text.strip().split("\n"):
        if ":" in line:
            # Split on first ":" to get "Row N" prefix, then items part
            # But items may contain ":" (e.g. "age:15") so we split on ": " (with space)
            prefix_and_items = line.split(": ", 1)
            if len(prefix_and_items) == 2:
                items_part = prefix_and_items[1]
                items = [x.strip() for x in items_part.split(",") if x.strip()]
                transactions.append(items)
    return transactions

print(f"📥 Loading eval dataset: {HF_EVAL_DATASET}")
ds = load_dataset(HF_EVAL_DATASET, split="train")
print(f"   Total: {len(ds)} eval datasets available")

# Sample N_EVAL from available
indices = random.sample(range(len(ds)), min(N_EVAL, len(ds)))

eval_samples = []
for i in indices:
    row = ds[i]
    csv_text = row["csv_text"]
    transactions = parse_csv_text_to_transactions(csv_text)
    ground_truth = json.loads(row["ground_truth"])

    eval_samples.append({
        "name"         : row["filename"],
        "csv_text"     : csv_text,
        "transactions" : transactions,
        "min_support"  : row["min_support"],
        "ground_truth" : ground_truth,   # Pre-computed Apriori ground truth
    })

print(f"✅ Loaded {len(eval_samples)} eval datasets (from {len(ds)} available)")

# Preview
s = eval_samples[0]
print(f"\nFirst sample: {s['name']}")
print(f"  Rows: {len(s['transactions'])}, Ground truth: {len(s['ground_truth'])} itemsets")
print(f"  Preview: {s['csv_text'][:200]}...")

In [ ]:
# ── Cell 6: Helper functions (Apriori + inference + metrics) ──────────────────

# ── System prompt (must match training EXACTLY) ───────────────────────────────
SYSTEM_PROMPT = (
    "You are a frequent itemset extractor. Given CSV transaction data and a "
    "minimum support count, identify all itemsets whose items co-occur in at "
    "least that many rows.\n\n"
    "Rules:\n"
    "1. Scan single items, pairs, and triples (up to size 3)\n"
    "2. Count = number of distinct rows containing ALL items in the itemset\n"
    "3. Only report itemsets with count >= min_support\n"
    "4. Canonicalize items: lowercase, trimmed, sorted alphabetically\n"
    '5. Row references: "Row N" format, 1-based indexing\n\n'
    "Think step by step inside <think> tags, then output ONLY a JSON array:\n"
    '[{"itemset": ["item1", "item2"], "count": N, "rows": ["Row 1", "Row 3"]}]'
)


def apriori(transactions: list, min_support: int = 3, max_size: int = 3) -> list:
    """Deterministic Apriori — ground truth."""
    if not transactions:
        return []
    row_labels = [f"Row {i+1}" for i in range(len(transactions))]
    counts = {}
    for idx, trans in enumerate(transactions):
        seen = set()
        for item in trans:
            item_n = str(item).strip().lower()
            if not item_n or item_n in seen:
                continue
            seen.add(item_n)
            k = (item_n,)
            if k not in counts:
                counts[k] = {"count": 0, "rows": []}
            counts[k]["count"] += 1
            counts[k]["rows"].append(row_labels[idx])

    def prune(d):
        return {k: v for k, v in d.items() if v["count"] >= min_support}

    L1 = prune(counts)
    if not L1:
        return []

    freq_levels = [L1]
    current = L1
    k = 2
    while k <= max_size and current:
        prev_keys = sorted(current.keys())
        candidates = set()
        for i in range(len(prev_keys)):
            for j in range(i + 1, len(prev_keys)):
                a, b = prev_keys[i], prev_keys[j]
                if a[:k-2] == b[:k-2]:
                    merged = tuple(sorted(set(a) | set(b)))
                    if len(merged) == k:
                        if all(tuple(sorted(sub)) in current
                               for sub in itertools.combinations(merged, k-1)):
                            candidates.add(merged)
        if not candidates:
            break
        cand_counts = {c: {"count": 0, "rows": []} for c in candidates}
        for idx, trans in enumerate(transactions):
            tset = {str(x).strip().lower() for x in trans}
            for cand in candidates:
                if set(cand).issubset(tset):
                    cand_counts[cand]["count"] += 1
                    cand_counts[cand]["rows"].append(row_labels[idx])
        current = prune(cand_counts)
        if current:
            freq_levels.append(current)
        k += 1

    out = []
    for level in freq_levels:
        for itemset, info in level.items():
            out.append({"itemset": list(itemset), "count": info["count"], "rows": info["rows"]})
    return out


def run_inference(csv_text: str, min_support: int) -> dict:
    """Run model and return parsed itemsets + metadata."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": (
            f"Find all frequent itemsets with minimum support count = {min_support} "
            f"in this dataset:\n\n{csv_text}"
        )},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    )
    if torch.cuda.is_available():
        inputs = inputs.to("cuda")

    with torch.no_grad():
        out = model.generate(
            input_ids=inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    raw = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

    # Strip <think> block
    has_think = "<think>" in raw and "</think>" in raw
    json_text = raw
    if has_think:
        json_text = raw.split("</think>", 1)[1].strip()

    # Parse JSON (3-level fallback)
    parsed, parse_ok = [], False
    for text in [json_text, raw]:
        try:
            parsed = json.loads(text)
            parse_ok = True
            break
        except json.JSONDecodeError:
            m = re.search(r"\[.*\]", text, re.DOTALL)
            if m:
                try:
                    parsed = json.loads(m.group())
                    parse_ok = True
                    break
                except json.JSONDecodeError:
                    pass

    # Normalize itemsets
    items = []
    for rec in (parsed if isinstance(parsed, list) else []):
        if not isinstance(rec, dict):
            continue
        itemset = rec.get("itemset", [])
        if not isinstance(itemset, list) or not itemset:
            continue
        items.append({
            "itemset": sorted(str(x).strip().lower() for x in itemset),
            "count"  : rec.get("count", 0),
            "rows"   : rec.get("rows", []),
        })

    return {"raw": raw, "items": items, "parse_ok": parse_ok, "has_think": has_think}


def canon(itemset):
    return frozenset(str(x).strip().lower() for x in itemset)


def compute_metrics(apriori_sets, model_sets, all_csv_items):
    """Compute all 7 eval metrics."""
    apr_c = {canon(s["itemset"]) for s in apriori_sets}
    mod_c = {canon(s["itemset"]) for s in model_sets}

    tp = apr_c & mod_c
    fp = mod_c - apr_c
    fn = apr_c - mod_c

    prec  = len(tp) / len(mod_c) if mod_c else 0.0
    rec   = len(tp) / len(apr_c) if apr_c else 0.0
    f1    = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0

    # Hallucination: itemsets containing items not in CSV
    halluc = sum(
        1 for s in model_sets
        if any(item not in all_csv_items for item in s["itemset"])
    )
    halluc_rate = halluc / len(model_sets) if model_sets else 0.0

    return {
        "apriori": len(apr_c), "model": len(mod_c),
        "tp": len(tp), "fp": len(fp), "fn": len(fn),
        "precision": round(prec, 4), "recall": round(rec, 4), "f1": round(f1, 4),
        "exact_match": f1 == 1.0, "hallucination": round(halluc_rate, 4),
    }


print("✅ Helper functions defined")

In [ ]:
# ── Cell 7: Run evaluation loop ────────────────────────────────────────────────

results = []
total_start = time.time()

for i, sample in enumerate(eval_samples, 1):
    ms = sample.get("min_support", MIN_SUPPORT)
    print(f"[{i}/{len(eval_samples)}] {sample['name']}  (min_support={ms})", end="  ", flush=True)
    t0 = time.time()

    # Ground truth — use pre-computed from HF dataset (verified Apriori)
    if "ground_truth" in sample and sample["ground_truth"]:
        apr_sets = sample["ground_truth"]
    else:
        apr_sets = apriori(sample["transactions"], ms, MAX_SIZE)

    # Model inference
    inf = run_inference(sample["csv_text"], ms)
    elapsed = time.time() - t0

    # Items that exist in the CSV
    all_items = {str(x).strip().lower() for trans in sample["transactions"] for x in trans}

    m = compute_metrics(apr_sets, inf["items"], all_items)

    print(
        f"F1={m['f1']:.0%}  P={m['precision']:.0%}  R={m['recall']:.0%}  "
        f"JSON={'✅' if inf['parse_ok'] else '❌'}  "
        f"think={'✅' if inf['has_think'] else '❌'}  "
        f"{elapsed:.0f}s"
    )

    results.append({
        "name"        : sample["name"],
        "rows"        : len(sample["transactions"]),
        "min_support" : ms,
        "apriori"     : m["apriori"],
        "model"       : m["model"],
        "TP"          : m["tp"],
        "FP"          : m["fp"],
        "FN"          : m["fn"],
        "precision"   : m["precision"],
        "recall"      : m["recall"],
        "f1"          : m["f1"],
        "exact_match" : m["exact_match"],
        "hallucination": m["hallucination"],
        "parse_ok"    : inf["parse_ok"],
        "has_think"   : inf["has_think"],
        "time_s"      : round(elapsed, 1),
        "_raw_output" : inf["raw"],
    })

print(f"\n✅ Finished {len(results)} datasets in {time.time() - total_start:.0f}s")

In [ ]:
# ── Cell 8: Display results ────────────────────────────────────────────────────

df = pd.DataFrame([{k: v for k, v in r.items() if k != "_raw_output"} for r in results])

# ── Per-dataset table ─────────────────────────────────────────────────────────
display_cols = ["name", "rows", "apriori", "model", "TP", "FP", "FN",
                "precision", "recall", "f1", "exact_match", "hallucination",
                "parse_ok", "has_think", "time_s"]

styled = df[display_cols].style.format({
    "precision"   : "{:.0%}",
    "recall"      : "{:.0%}",
    "f1"          : "{:.0%}",
    "hallucination": "{:.0%}",
    "time_s"      : "{:.0f}s",
}).background_gradient(subset=["f1"], cmap="RdYlGn", vmin=0, vmax=1)

display(styled)

# ── Aggregate summary ─────────────────────────────────────────────────────────
n = len(df)
summary = {
    "Model"             : MODEL_PATH,
    "Datasets evaluated": n,
    "Avg Precision"     : f"{df['precision'].mean():.1%}",
    "Avg Recall"        : f"{df['recall'].mean():.1%}",
    "Avg F1"            : f"{df['f1'].mean():.1%}",
    "Exact Match"       : f"{df['exact_match'].sum()}/{n}  ({df['exact_match'].mean():.1%})",
    "JSON Parse Rate"   : f"{df['parse_ok'].mean():.1%}",
    "Think Rate"        : f"{df['has_think'].mean():.1%}",
    "Hallucination"     : f"{df['hallucination'].mean():.1%}",
    "Avg Inference Time": f"{df['time_s'].mean():.1f}s",
}

print("\n" + "═" * 50)
print("  EVALUATION SUMMARY")
print("═" * 50)
for k, v in summary.items():
    print(f"  {k:<22} {v}")

# ── Pass/fail verdict ─────────────────────────────────────────────────────────
avg_f1    = df['f1'].mean()
parse_rt  = df['parse_ok'].mean()
halluc    = df['hallucination'].mean()
passed    = avg_f1 >= 0.80 and parse_rt >= 0.90 and halluc <= 0.05
print("─" * 50)
if passed:
    print("  🎉 PASSED — Model meets production targets!")
else:
    reasons = []
    if avg_f1   < 0.80: reasons.append(f"F1 {avg_f1:.1%} < 80%")
    if parse_rt < 0.90: reasons.append(f"Parse {parse_rt:.1%} < 90%")
    if halluc   > 0.05: reasons.append(f"Halluc {halluc:.1%} > 5%")
    print(f"  ⚠️  NOT YET — {', '.join(reasons)}")
print("═" * 50)

In [ ]:
# ── Cell 9 (optional): Inspect a specific result ──────────────────────────────
# Change idx to inspect any dataset

idx = 0   # ← change this

r = results[idx]
print(f"Dataset: {r['name']}")
print(f"F1={r['f1']:.0%}  P={r['precision']:.0%}  R={r['recall']:.0%}")
print(f"Apriori: {r['apriori']} itemsets | Model: {r['model']} itemsets")
print(f"TP={r['TP']} FP={r['FP']} FN={r['FN']}")
print(f"JSON parse: {'✅' if r['parse_ok'] else '❌'}")
print(f"Has <think>: {'✅' if r['has_think'] else '❌'}")
print()
print("─── Raw model output ───────────────────────────")
print(r['_raw_output'][:2000])

In [ ]:
# ── Cell 10: Save results ──────────────────────────────────────────────────────

import os, json
from pathlib import Path
from datetime import datetime, timezone

out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M")

# Summary JSON
summary_dict = {
    "model"       : MODEL_PATH,
    "timestamp"   : datetime.now(timezone.utc).isoformat(),
    "n_datasets"  : len(df),
    "avg_f1"      : round(float(df["f1"].mean()), 4),
    "avg_precision": round(float(df["precision"].mean()), 4),
    "avg_recall"  : round(float(df["recall"].mean()), 4),
    "exact_match_rate": round(float(df["exact_match"].mean()), 4),
    "parse_rate"  : round(float(df["parse_ok"].mean()), 4),
    "think_rate"  : round(float(df["has_think"].mean()), 4),
    "hallucination_rate": round(float(df["hallucination"].mean()), 4),
    "avg_time_s"  : round(float(df["time_s"].mean()), 1),
}
(out_dir / f"summary_{timestamp}.json").write_text(
    json.dumps(summary_dict, indent=2), encoding="utf-8"
)

# Full results CSV
df_save = df.drop(columns=["_raw_output"] if "_raw_output" in df.columns else [])
csv_path = out_dir / f"results_{timestamp}.csv"
df_save.to_csv(csv_path, index=False)

# Raw outputs (one file each — useful for debugging)
raw_dir = out_dir / f"raw_{timestamp}"
raw_dir.mkdir(exist_ok=True)
for r in results:
    (raw_dir / f"{r['name']}.txt").write_text(r["_raw_output"], encoding="utf-8")

print(f"✅ Saved to {out_dir}/")
print(f"   summary_{timestamp}.json")
print(f"   results_{timestamp}.csv")
print(f"   raw_{timestamp}/  ({len(results)} raw output files)")